# Phase 1 Data Pipeline (MIND-small)

## Scope Lock
- Phase 1 only: Data Pipeline + Feature Engineering.
- MIND-small only. No MIND-large processing in this notebook.
- No causal estimation/refutation cells and no RL cells.

## Contract Locks From Documentation
- `A=1` for shown items; sampled controls use `A=0` and `Y_click=0`.
- `Y_diversity = 1 - cosine(U_history_emb, I_title_emb)`.
- Negative sampling target ratio: `1:4` (shown:negative).
- PCA requirement: `n_components=32` for tabular compatibility.

## User-Approved Overrides
- `U_history_emb` is locked to 768 dimensions for this implementation.
- Split strategy is by `ImpressionID` session boundary with `70/15/15` train/val/test.

## Outputs
- `data/scm_train.parquet`
- `data/scm_val.parquet`
- `data/scm_test.parquet`
- `data/processed/phase1_data_report.json`
- `data/interim/scm_full_embeddings.parquet` (full embeddings sidecar)


In [ ]:
import sys
import os
import json
import logging
from pathlib import Path
import random
from datetime import datetime

import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

_cwd = Path.cwd()
_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.config import *
from src.data_pipeline.io_utils import *
from src.data_pipeline.nlp_utils import *
from src.data_pipeline.parsers import *
from src.data_pipeline.embedder import build_title_encoder
from src.data_pipeline.features import *
from src.data_pipeline.scm_builder import *
from src.data_pipeline.streaming import *
from src.gpu_utils import gpu_available

%load_ext autoreload
%autoreload 2

PROJECT_ROOT = _root
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
RAW_SMALL_DIR = RAW_DIR / "MIND-small"

for path in [DATA_DIR, RAW_DIR, INTERIM_DIR, PROCESSED_DIR, NOTEBOOKS_DIR,
             RAW_SMALL_DIR, RAW_SMALL_TRAIN_DIR, RAW_SMALL_DEV_DIR, RAW_SMALL_TEST_DIR]:
    path.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root: {DATA_DIR}")
print(f"GPU acceleration: {'ENABLED' if gpu_available() else 'UNAVAILABLE (CPU fallback)'}")


## Execution (Top-Down Deterministic Flow)

Run all execution cells from top to bottom with fixed seed.

### 1_load_news

In [ ]:
external_candidates = [
    PROJECT_ROOT / "MIND small dataset",
]

dataset_files = prepare_mind_small_dataset(RAW_SMALL_DIR, external_candidates)
print("Resolved MIND-small files:")
for key, value in dataset_files.items():
    print(f"  {key}: {value}")

news_paths = [dataset_files["train_news"], dataset_files["dev_news"]]
if "test_news" in dataset_files:
    news_paths.append(dataset_files["test_news"])

news_df = load_news_frames(news_paths)
print(f"News rows: {news_df.shape[0]:,} | Unique items: {news_df['NewsID'].nunique():,}")
news_df.head(3)

### 2_news_features

In [ ]:
title_encoder, encoder_meta = build_title_encoder(TITLE_EMBED_MODEL, TITLE_EMBED_DIM)
sentiment_analyzer = build_sentiment_analyzer()

news_features_df = compute_news_features(
    news_df=news_df,
    title_encoder=title_encoder,
    sentiment_analyzer=sentiment_analyzer,
    entity_dim=ENTITY_EMBED_DIM,
)

print("Encoder metadata:")
print(json.dumps(encoder_meta, indent=2))
print(f"News feature rows: {news_features_df.shape[0]:,}")
news_features_df.head(3)

### 3_load_behaviors

In [ ]:
behavior_paths = [dataset_files["train_behaviors"], dataset_files["dev_behaviors"]]
if "test_behaviors" in dataset_files:
    behavior_paths.append(dataset_files["test_behaviors"])

scm_parts_dir = DATA_DIR / "scm_parts"
if scm_parts_dir.exists():
    import shutil
    shutil.rmtree(scm_parts_dir)
    
print(f"Streaming behaviors to partitioned parquet at {scm_parts_dir}...")
stats = stream_and_build(
    behavior_paths=behavior_paths,
    news_features_df=news_features_df,
    output_dir=scm_parts_dir,
    chunksize=2000,
    neg_ratio=NEG_RATIO,
    seed=SEED
)

print("Streaming build complete!")
print(json.dumps(stats, indent=2))

In [ ]:
scm_parts_dir = DATA_DIR / "scm_parts"
import pyarrow.parquet as pq

def _load_fragments(pattern: str) -> pd.DataFrame:
    """Load parquet fragments one file at a time to stay under 2GB/alloc."""
    import gc
    frames = []
    for f in sorted(scm_parts_dir.rglob(pattern)):
        tbl = pq.read_table(f)
        df = tbl.to_pandas()
        frames.append(df)
        del tbl, df
        gc.collect()
    result = pd.concat(frames, ignore_index=True)
    del frames
    gc.collect()
    return result

full_df = _load_fragments("*.parquet")
print(f"Total SCM rows loaded: {len(full_df):,}")

In [ ]:
required_cols = ['user_id', 'item_id', 'impression_id', 'A', 'Y_click',
                 'Y_diversity', 'U_dwell_mean', 'I_category', 'I_sentiment']

reduced_df, pca_report = reduce_embedding_columns(full_df, PCA_COMPONENTS, SEED)
print('PCA report:', json.dumps(pca_report, indent=2))

train_df, val_df, test_df, split_report = split_by_impression_id(
    reduced_df, SPLIT_RATIOS, SEED
)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

quality_report = run_quality_checks(train_df, val_df, test_df, required_cols, NEG_RATIO)
print('Quality checks passed.')

save_parquet(train_df, DATA_DIR / 'scm_train.parquet')
save_parquet(val_df, DATA_DIR / 'scm_val.parquet')
save_parquet(test_df, DATA_DIR / 'scm_test.parquet')
print('Saved scm_train/val/test.parquet')

In [ ]:
print(json.dumps(quality_report, indent=2))
print(f"\nSplit sizes: train={len(train_df):,} val={len(val_df):,} test={len(test_df):,}")
print(f"Treatment ratio: {quality_report['observed_control_per_treatment']:.2f}")
train_df.head(5)

In [ ]:
output_paths = {
    'scm_train_parquet': str(DATA_DIR / 'scm_train.parquet'),
    'scm_val_parquet': str(DATA_DIR / 'scm_val.parquet'),
    'scm_test_parquet': str(DATA_DIR / 'scm_test.parquet'),
}
full_df.to_parquet(INTERIM_DIR / 'scm_full_embeddings.parquet', index=False)

phase1_report = build_phase1_report(
    train_df, val_df, test_df, split_report, quality_report, pca_report,
    encoder_meta, output_paths
)
with open(PROCESSED_DIR / 'phase1_data_report.json', 'w') as f:
    json.dump(phase1_report, f, indent=2)

print("Primary notebook:")
print(f"  {NOTEBOOKS_DIR / 'phase_1_data_pipeline_mind_small.ipynb'}")
print("\nArtifacts:")
for key, value in output_paths.items():
    print(f"  {key}: {value}")
print("\nPhase 1 complete.")

### 4_inspect_partitioned_dataset

## Phase 1 Outputs